# Predictive Layer — Where will the next pothole form?

Trains a LightGBM regressor on a 250m grid over Bangalore using:
- historical complaint density per cell (from `data/raw/icmc.json`)
- IMD daily rainfall (synthesized here — replace with real IMD CSV)
- proxy traffic volume (population density approximation)
- road-age proxy (uniform random for the synthetic version)

Output: per-cell risk score for the next 30 days → exported as GeoJSON heatmap
consumed by the Next.js `/map` page.

In [ ]:
import json, pathlib, sys, math, random
from datetime import datetime, timedelta
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(7)
BLR_BBOX = (12.85, 13.10, 77.45, 77.75)  # min_lat, max_lat, min_lng, max_lng
CELL_DEG = 0.0025  # ~250m at this latitude

lats = np.arange(BLR_BBOX[0], BLR_BBOX[1], CELL_DEG)
lngs = np.arange(BLR_BBOX[2], BLR_BBOX[3], CELL_DEG)
grid = pd.DataFrame([(la, ln) for la in lats for ln in lngs], columns=['lat','lng'])
print(f'{len(grid):,} cells covering Bangalore at {CELL_DEG*111:.0f}m resolution')

In [ ]:
# Feature engineering — synthetic versions; swap with real data once available.
city_lat, city_lng = 12.9716, 77.5946
grid['dist_from_centre_km'] = np.hypot(grid.lat - city_lat, grid.lng - city_lng) * 111

grid['population_proxy'] = np.clip(1500 * np.exp(-grid.dist_from_centre_km / 6) + rng.normal(0, 200, len(grid)), 50, 4000)
grid['road_age_years']   = rng.integers(2, 25, len(grid))
grid['rain_30d_mm']      = rng.normal(180, 60, len(grid)).clip(0)
grid['hist_complaints_180d'] = rng.poisson(grid.population_proxy / 400)
grid['near_drain_meters'] = rng.uniform(5, 800, len(grid))

# Synthetic ground-truth: cells with high rainfall × old roads × high traffic spawn potholes.
grid['future_potholes_30d'] = (
    0.012 * grid.rain_30d_mm
    + 0.18  * grid.road_age_years
    + 0.003 * grid.population_proxy
    + 0.4   * grid.hist_complaints_180d
    - 0.002 * grid.near_drain_meters
    + rng.normal(0, 1.5, len(grid))
).clip(0)

grid.describe().round(2)

In [ ]:
FEATURES = ['lat','lng','dist_from_centre_km','population_proxy','road_age_years',
            'rain_30d_mm','hist_complaints_180d','near_drain_meters']
TARGET = 'future_potholes_30d'

X_tr, X_te, y_tr, y_te = train_test_split(grid[FEATURES], grid[TARGET], test_size=0.2, random_state=7)

model = lgb.LGBMRegressor(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_child_samples=20, subsample=0.9, colsample_bytree=0.9,
    random_state=7,
)
model.fit(X_tr, y_tr, eval_set=[(X_te, y_te)], callbacks=[lgb.early_stopping(20)])

preds = model.predict(X_te)
print(f'MAE  : {mean_absolute_error(y_te, preds):.3f}')
print(f'R²   : {r2_score(y_te, preds):.3f}')

In [ ]:
# Top-K hotspot extraction for the heatmap.
grid['risk_score'] = model.predict(grid[FEATURES])
grid['risk_norm']  = (grid.risk_score - grid.risk_score.min()) / (grid.risk_score.max() - grid.risk_score.min() + 1e-9)
top = grid.nlargest(150, 'risk_score').copy()

geojson = {
    'type': 'FeatureCollection',
    'features': [
        {
            'type': 'Feature',
            'geometry': {'type': 'Point', 'coordinates': [float(r.lng), float(r.lat)]},
            'properties': {
                'risk': round(float(r.risk_norm), 3),
                'predicted_30d': round(float(r.risk_score), 2),
                'drivers': {
                    'rain_30d_mm': round(float(r.rain_30d_mm), 1),
                    'road_age': int(r.road_age_years),
                    'hist_complaints': int(r.hist_complaints_180d),
                },
            },
        }
        for r in top.itertuples()
    ],
}
out = pathlib.Path('../data/processed/hotspots.geojson')
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(geojson))
print(f'wrote {len(top)} hotspots → {out}')

In [ ]:
# Feature importance — useful for the pitch slide.
imp = pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_}).sort_values('importance', ascending=False)
imp